# Fabric Unified Monitoring & Analytics

**Enterprise Capacity Intelligence — Combining Capacity Metrics + Chargeback Reporting**

This notebook bridges the two core Microsoft Fabric monitoring apps — the **Capacity Metrics App** (105 tables, 294 measures) and the **Chargeback Reporting App** (10 tables, 31 measures) — to answer questions that neither can answer alone.

---

## What This Solves

| # | Problem | Why Neither App Alone Can Answer It |
|---|---------|-------------------------------------|
| 1 | **Who caused this throttling?** | Capacity Metrics shows *that* throttling happened. Chargeback shows *who* consumed CUs. Neither connects the two. |
| 2 | **Upgrade SKU or fix bad actors?** | The most expensive question. Requires correlating per-item/user CU burn (Chargeback) with throttling timestamps (Capacity Metrics). |
| 3 | **Total cost of a workspace** | Compute cost (CU-seconds) is in Chargeback. Storage cost (GB) is in Capacity Metrics. No combined view exists. |
| 4 | **Stale items burning capacity** | Items with scheduled refreshes (high CU in Chargeback) but zero downstream users need to be identified. |
| 5 | **Which department is killing the capacity?** | Chargeback has Domains/Subdomains. Capacity Metrics has throttling events. Neither connects them. |
| 6 | **Optimal refresh scheduling** | Requires overlaying CU load curves (Capacity Metrics 30-sec data) with heavy background operations (Chargeback user/item data). |
| 7 | **Surge protection root cause** | New v54 tables show blocked workspaces, but not the *users* whose operations triggered the surge. Chargeback fills this gap. |

---

## Data Sources

| Model | Workspace | Dataset ID | Tables | Measures |
|-------|-----------|------------|--------|----------|
| Capacity Metrics App (v54) | `87d74804-68a9-4e43-8742-4d295cecd2ea` | `1d1a8613-f490-4ba8-9bb4-72324c3a747e` | 105 | 294 |
| Chargeback Reporting | `6741112c-52c2-42e4-8e6f-5e5159d28111` | `1df83a07-9420-4ac3-9e7c-90853fb6d238` | 10 | 31 |

## Join Keys

| Key | Capacity Metrics Column | Chargeback Column | Join Type |
|-----|------------------------|-------------------|-----------|
| Capacity | `Capacities[Capacity Id]` | `Capacities[Capacity Id]` | Direct |
| Item | `Items[Item Id]` | `Items[Item Id]` | Direct |
| Workspace | `Workspaces[Workspace Id]` | `Workspaces[Workspace Id]` | Direct |
| Date | `Dates[Date]` | `Dates[Date]` | Direct |
| Operation | `Operation Names` / `Items Operations[Operation name]` | `Chargeback[Operation name]` | Direct |
| Billing Type | `Billing Type` | `Chargeback[Billing type]` | Direct |
| User | `Timepoint Interactive Detail[User]` | `Chargeback[User]` | Direct |
| Experience/Workload | `Item History Main[Experience]` / `Items Operations[Item kind]` | `Chargeback[Experience]` | Direct |

## 0. Setup & Configuration

In [ ]:
# ============================================================================
# CONFIGURATION — Update these values for your environment
# ============================================================================

# Capacity Metrics App
CM_WORKSPACE_ID = "87d74804-68a9-4e43-8742-4d295cecd2ea"
CM_DATASET_ID = "1d1a8613-f490-4ba8-9bb4-72324c3a747e"

# Chargeback Reporting App
CB_WORKSPACE_ID = "6741112c-52c2-42e4-8e6f-5e5159d28111"
CB_DATASET_ID = "1df83a07-9420-4ac3-9e7c-90853fb6d238"

# Target workspace for outputs
TARGET_WORKSPACE_ID = "5bcbe629-5529-40ac-8909-9b155417ecd5"  # MonitoringAdmin

# Power BI REST API base
PBI_API_BASE = "https://api.powerbi.com/v1.0/myorg"
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"

In [ ]:
# ============================================================================
# DEPENDENCIES
# ============================================================================
import json
import time
import warnings
from datetime import datetime, timedelta
from typing import Optional

import pandas as pd
import requests

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

print(f"pandas {pd.__version__} loaded")

In [ ]:
# ============================================================================
# AUTHENTICATION
# Works in: Fabric notebooks (automatic), local with azure-identity
# ============================================================================

def get_token() -> str:
    """Get Power BI access token. Tries Fabric sempy first, falls back to azure-identity."""
    # Fabric notebook environment
    try:
        from notebookutils import mssparkutils
        return mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
    except ImportError:
        pass

    # Local / Azure CLI / Managed Identity
    from azure.identity import DefaultAzureCredential
    credential = DefaultAzureCredential()
    token = credential.get_token("https://analysis.windows.net/powerbi/api/.default")
    return token.token

def get_headers() -> dict:
    return {"Authorization": f"Bearer {get_token()}", "Content-Type": "application/json"}

# Verify auth
_h = get_headers()
print("Authentication successful")

In [ ]:
# ============================================================================
# QUERY ENGINE — Robust DAX query executor with retry and pagination
# ============================================================================

def execute_dax(
    workspace_id: str,
    dataset_id: str,
    dax_query: str,
    max_retries: int = 3,
) -> pd.DataFrame:
    """Execute a DAX query against a Power BI dataset and return a DataFrame.
    
    Handles:
    - 429 rate limiting with retry-after
    - Token refresh on 401
    - Clean column name extraction
    """
    url = f"{PBI_API_BASE}/groups/{workspace_id}/datasets/{dataset_id}/executeQueries"
    payload = {
        "queries": [{"query": dax_query}],
        "serializerSettings": {"includeNulls": True},
    }

    for attempt in range(max_retries):
        headers = get_headers()
        resp = requests.post(url, headers=headers, json=payload)

        if resp.status_code == 200:
            result = resp.json()
            if "error" in result:
                raise RuntimeError(f"DAX error: {result['error']}")
            rows = result["results"][0]["tables"][0]["rows"]
            if not rows:
                return pd.DataFrame()
            df = pd.DataFrame(rows)
            # Clean column names: '[Table[Col]]' -> 'Col'
            df.columns = [c.strip("[]").split("[")[-1].rstrip("]") for c in df.columns]
            return df

        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 30))
            print(f"  Rate limited. Waiting {wait}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)
            continue

        if resp.status_code == 401 and attempt < max_retries - 1:
            print("  Token expired, refreshing...")
            continue

        resp.raise_for_status()

    raise RuntimeError(f"Failed after {max_retries} attempts")


def query_cm(dax: str) -> pd.DataFrame:
    """Shorthand: query Capacity Metrics model."""
    return execute_dax(CM_WORKSPACE_ID, CM_DATASET_ID, dax)


def query_cb(dax: str) -> pd.DataFrame:
    """Shorthand: query Chargeback model."""
    return execute_dax(CB_WORKSPACE_ID, CB_DATASET_ID, dax)


# Verify connectivity to both models
_test_cm = query_cm('EVALUATE ROW("status", "ok")')
_test_cb = query_cb('EVALUATE ROW("status", "ok")')
print(f"Capacity Metrics: connected | Chargeback: connected")

---
## 1. Load Dimension Tables (Shared Context)

Both models share the same dimension keys. We load them once and use them as the backbone for all cross-model joins.

In [ ]:
# ============================================================================
# DIMENSION TABLES — Load from both models, deduplicate
# ============================================================================

# Capacities (from CM — has more columns)
dim_capacities = query_cm("""
    EVALUATE
    SELECTCOLUMNS(
        'Capacities',
        "Capacity Id", 'Capacities'[Capacity Id],
        "Capacity Name", 'Capacities'[Capacity name],
        "SKU", 'Capacities'[SKU],
        "Region", 'Capacities'[Region],
        "State", 'Capacities'[State]
    )
""")

# Add core count from Chargeback (CM doesn't have it)
cb_caps = query_cb("""
    EVALUATE
    SELECTCOLUMNS(
        'Capacities',
        "Capacity Id", 'Capacities'[Capacity Id],
        "Core count", 'Capacities'[Core count]
    )
""")
dim_capacities = dim_capacities.merge(cb_caps, on="Capacity Id", how="left")

# Items (from CM — has more columns)
dim_items = query_cm("""
    EVALUATE
    SELECTCOLUMNS(
        'Items',
        "Item Id", 'Items'[Item Id],
        "Item Name", 'Items'[Item name],
        "Item Kind", 'Items'[Item kind],
        "Workspace Id", 'Items'[Workspace Id],
        "Workspace Name", 'Items'[Workspace name],
        "Capacity Id", 'Items'[Capacity Id],
        "Billable Type", 'Items'[Billable type]
    )
""")

# Workspaces
dim_workspaces = query_cm("""
    EVALUATE
    SELECTCOLUMNS(
        'Workspaces',
        "Workspace Id", 'Workspaces'[Workspace Id],
        "Workspace Name", 'Workspaces'[Workspace name],
        "Capacity Id", 'Workspaces'[Capacity Id]
    )
""")

# Domains (Chargeback only — CM doesn't have this)
dim_domains = query_cb("""
    EVALUATE
    SELECTCOLUMNS(
        'Domains',
        "Domain unique key", 'Domains'[Domain unique key],
        "Domain Id", 'Domains'[Domain Id],
        "Domain", 'Domains'[Domain],
        "Subdomain Id", 'Domains'[Subdomain Id],
        "Subdomain", 'Domains'[Subdomain]
    )
""")

print(f"Dimensions loaded:")
print(f"  Capacities: {len(dim_capacities)} rows")
print(f"  Items:      {len(dim_items)} rows")
print(f"  Workspaces: {len(dim_workspaces)} rows")
print(f"  Domains:    {len(dim_domains)} rows")
dim_capacities.head(3)

---
## 2. Load Fact Tables

### 2a. Chargeback Facts (Daily grain — user, item, operation, CU, duration)

In [ ]:
# ============================================================================
# CHARGEBACK FACTS — Daily per-user/item/operation CU consumption
# ============================================================================

df_chargeback = query_cb("""
    EVALUATE
    SELECTCOLUMNS(
        'Chargeback',
        "Capacity Id", 'Chargeback'[Capacity Id],
        "Date", 'Chargeback'[Date],
        "Item Id", 'Chargeback'[Item Id],
        "Workspace Id", 'Chargeback'[Workspace Id],
        "Billing type", 'Chargeback'[Billing type],
        "Operation name", 'Chargeback'[Operation name],
        "User", 'Chargeback'[User],
        "CU (s)", 'Chargeback'[CU (s)],
        "Duration (s)", 'Chargeback'[Duration (s)],
        "Operations", 'Chargeback'[Operations],
        "Experience", 'Chargeback'[Experience],
        "Domain unique key", 'Chargeback'[Domain unique key]
    )
""")

df_chargeback["Date"] = pd.to_datetime(df_chargeback["Date"])

# Enrich with domain names
df_chargeback = df_chargeback.merge(dim_domains, on="Domain unique key", how="left")

print(f"Chargeback facts: {len(df_chargeback):,} rows | Date range: {df_chargeback['Date'].min()} to {df_chargeback['Date'].max()}")
df_chargeback.head(3)

### 2b. Capacity Metrics — Usage Summary (pre-aggregated health metrics)

In [ ]:
# ============================================================================
# USAGE SUMMARY (Last 7 days) — Hourly capacity health snapshot
# New in v54: pre-aggregated utilization, throttling risk, cumulative debt
# ============================================================================

df_usage_7d = query_cm("""
    EVALUATE
    SELECTCOLUMNS(
        'Usage Summary (Last 7 days)',
        "Timestamp", 'Usage Summary (Last 7 days)'[Timestamp],
        "Capacity Id", 'Usage Summary (Last 7 days)'[Capacity Id],
        "CU (s)", 'Usage Summary (Last 7 days)'[CU (s)],
        "Average CU %", 'Usage Summary (Last 7 days)'[Average CU %],
        "Maximum CU %", 'Usage Summary (Last 7 days)'[Maximum CU %],
        "Interactive rejection (s)", 'Usage Summary (Last 7 days)'[Interactive rejection (s)],
        "Background rejection (s)", 'Usage Summary (Last 7 days)'[Background rejection (s)],
        "Interactive delay (s)", 'Usage Summary (Last 7 days)'[Interactive delay (s)],
        "Average utilization", 'Usage Summary (Last 7 days)'[Average utilization]
    )
""")

df_usage_7d["Timestamp"] = pd.to_datetime(df_usage_7d["Timestamp"])
df_usage_7d["Date"] = df_usage_7d["Timestamp"].dt.date

print(f"Usage Summary (7d): {len(df_usage_7d):,} rows")
df_usage_7d.head(3)

### 2c. Capacity Metrics — Usage Operations (who/what is consuming CUs per hour)

In [ ]:
# ============================================================================
# USAGE OPERATIONS (Last 7 days) — Per-operation hourly breakdown
# ============================================================================

df_usage_ops_7d = query_cm("""
    EVALUATE
    SELECTCOLUMNS(
        'Usage Operation (Last 7 days)',
        "Capacity Id", 'Usage Operation (Last 7 days)'[Capacity Id],
        "Utilization type", 'Usage Operation (Last 7 days)'[Utilization type],
        "Hours", 'Usage Operation (Last 7 days)'[Hours],
        "CU (s)", 'Usage Operation (Last 7 days)'[CU (s)],
        "Duration (s)", 'Usage Operation (Last 7 days)'[Duration (s)],
        "Throttling (s)", 'Usage Operation (Last 7 days)'[Throttling (s)],
        "Users", 'Usage Operation (Last 7 days)'[Users],
        "TIMESTAMP", 'Usage Operation (Last 7 days)'[TIMESTAMP],
        "Failure operations", 'Usage Operation (Last 7 days)'[Failure operations],
        "Rejected operations", 'Usage Operation (Last 7 days)'[Rejected operations],
        "Successful operations", 'Usage Operation (Last 7 days)'[Successful operations]
    )
""")

df_usage_ops_7d["Hours"] = pd.to_datetime(df_usage_ops_7d["Hours"])

print(f"Usage Operations (7d): {len(df_usage_ops_7d):,} rows")
df_usage_ops_7d.head(3)

### 2d. Storage Facts

In [ ]:
# ============================================================================
# STORAGE BY WORKSPACES AND DAY — Daily storage utilization per workspace
#
# Note: This table uses 'PremiumCapacityId' (not 'Capacity Id') — verified
# via Scanner API. Column names are case-sensitive ('Operation name' not
# 'Operation Name').
# ============================================================================

try:
    df_storage = query_cm("""
        EVALUATE
        SELECTCOLUMNS(
            'Storage By Workspaces And Day',
            "Workspace Id", 'Storage By Workspaces And Day'[Workspace Id],
            "Capacity Id", 'Storage By Workspaces And Day'[PremiumCapacityId],
            "Date", 'Storage By Workspaces And Day'[Date],
            "Utilization (GB)", 'Storage By Workspaces And Day'[Utilization (GB)],
            "Operation Name", 'Storage By Workspaces And Day'[Operation name],
            "Workload Kind", 'Storage By Workspaces And Day'[Workload kind]
        )
    """)

    if not df_storage.empty:
        df_storage["Date"] = pd.to_datetime(df_storage["Date"])
        print(f"Storage By Workspaces And Day: {len(df_storage):,} rows")
        df_storage.head(3)
    else:
        print("Storage table returned 0 rows.")
except Exception as e:
    print(f"Storage query failed: {e}")
    print("Storage analyses will be skipped.")
    df_storage = pd.DataFrame(columns=["Workspace Id", "Capacity Id", "Date", "Utilization (GB)", "Operation Name", "Workload Kind"])

### 2e. Surge Protection (new in v54)

In [ ]:
# ============================================================================
# SURGE PROTECTION — Blocked workspaces with detail
# New in v54: shows which workspaces were blocked due to sustained overuse
# May return empty if no surge events occurred in the data window.
# ============================================================================

_surge_cols = ["Workspace Id", "Workspace name", "Blocked date", "Blocked end", "Blocked duration (hours)", "How blocked", "How Blocked Category", "Affected users", "Capacity Id", "Interactive rejected operations", "Background rejected operations"]

try:
    df_surge_detail = query_cm("""
        EVALUATE
        SELECTCOLUMNS(
            'Surge Protection Blocked Workspaces Detail',
            "Workspace Id", 'Surge Protection Blocked Workspaces Detail'[Workspace Id],
            "Workspace name", 'Surge Protection Blocked Workspaces Detail'[Workspace name],
            "Blocked date", 'Surge Protection Blocked Workspaces Detail'[Blocked date],
            "Blocked end", 'Surge Protection Blocked Workspaces Detail'[Blocked end],
            "Blocked duration (hours)", 'Surge Protection Blocked Workspaces Detail'[Blocked duration (hours)],
            "How blocked", 'Surge Protection Blocked Workspaces Detail'[How blocked],
            "How Blocked Category", 'Surge Protection Blocked Workspaces Detail'[How Blocked Category],
            "Affected users", 'Surge Protection Blocked Workspaces Detail'[Affected users],
            "Capacity Id", 'Surge Protection Blocked Workspaces Detail'[Capacity Id],
            "Interactive rejected operations", 'Surge Protection Blocked Workspaces Detail'[Interactive rejected operations],
            "Background rejected operations", 'Surge Protection Blocked Workspaces Detail'[Background rejected operations]
        )
    """)

    if not df_surge_detail.empty:
        df_surge_detail["Blocked date"] = pd.to_datetime(df_surge_detail["Blocked date"])
        df_surge_detail["Blocked end"] = pd.to_datetime(df_surge_detail["Blocked end"])

except Exception as e:
    print(f"Surge Protection query failed: {e}")
    df_surge_detail = pd.DataFrame(columns=_surge_cols)

print(f"Surge Protection Detail: {len(df_surge_detail):,} rows")
if not df_surge_detail.empty:
    df_surge_detail.head(5)

### 2f. Item History (new in v54 — historical CU/duration/throttling per item)

In [ ]:
# ============================================================================
# ITEM HISTORY — Detailed per-item operation history with CU and throttling
# New in v54: allows item-level trending over configurable date range
#
# These tables may return empty if the Item History parameters (StartHistory,
# EndHistory) haven't been set via the app UI. We handle that gracefully.
# ============================================================================

_hist_cols_main = ["WorkspaceName", "ArtifactName", "OperationName", "UtilizationType", "ArtifactKind", "Billing type", "Experience", "ItemHistoryUniquKey"]
_hist_cols_ops = ["Day", "Operations", "CU (s)", "Duration (s)", "Throttling (s)", "ItemHistoryUniquKey"]

try:
    df_item_hist_main = query_cm("""
        EVALUATE
        SELECTCOLUMNS(
            'Item History Main',
            "WorkspaceName", 'Item History Main'[WorkspaceName],
            "ArtifactName", 'Item History Main'[ArtifactName],
            "OperationName", 'Item History Main'[OperationName],
            "UtilizationType", 'Item History Main'[UtilizationType],
            "ArtifactKind", 'Item History Main'[ArtifactKind],
            "Billing type", 'Item History Main'[Billing type],
            "Experience", 'Item History Main'[Experience],
            "ItemHistoryUniquKey", 'Item History Main'[ItemHistoryUniquKey]
        )
    """)
except Exception as e:
    print(f"Item History Main query failed: {e}")
    df_item_hist_main = pd.DataFrame(columns=_hist_cols_main)

try:
    df_item_hist_ops = query_cm("""
        EVALUATE
        SELECTCOLUMNS(
            'Item History Operation',
            "Day", 'Item History Operation'[Day],
            "Operations", 'Item History Operation'[Operations],
            "CU (s)", 'Item History Operation'[CU (s)],
            "Duration (s)", 'Item History Operation'[Duration (s)],
            "Throttling (s)", 'Item History Operation'[Throttling (s)],
            "ItemHistoryUniquKey", 'Item History Operation'[ItemHistoryUniquKey]
        )
    """)
except Exception as e:
    print(f"Item History Operation query failed: {e}")
    df_item_hist_ops = pd.DataFrame(columns=_hist_cols_ops)

if not df_item_hist_ops.empty:
    df_item_hist_ops["Day"] = pd.to_datetime(df_item_hist_ops["Day"])

# Join only if both have data and the key column exists
if not df_item_hist_ops.empty and not df_item_hist_main.empty:
    df_item_history = df_item_hist_ops.merge(df_item_hist_main, on="ItemHistoryUniquKey", how="left")
else:
    df_item_history = pd.DataFrame(columns=_hist_cols_ops + [c for c in _hist_cols_main if c != "ItemHistoryUniquKey"])

print(f"Item History Main: {len(df_item_hist_main):,} rows")
print(f"Item History Operations: {len(df_item_hist_ops):,} rows")
print(f"Item History (joined): {len(df_item_history):,} rows")
if not df_item_history.empty:
    df_item_history.head(3)

### 2g. Items Operations (new in v54 — operation metadata per item)

In [ ]:
# ============================================================================
# ITEMS OPERATIONS — Distinct operations per item with user counts
# New in v54: includes Release type and operation-level granularity
# ============================================================================

df_items_ops = query_cm("""
    EVALUATE
    SELECTCOLUMNS(
        'Items Operations',
        "Item Id", 'Items Operations'[Item Id],
        "Workspace Id", 'Items Operations'[Workspace Id],
        "Capacity Id", 'Items Operations'[Capacity Id],
        "Operation name", 'Items Operations'[Operation name],
        "Item name", 'Items Operations'[Item name],
        "Workspace name", 'Items Operations'[Workspace name],
        "Item kind", 'Items Operations'[Item kind],
        "Users", 'Items Operations'[Users],
        "Release type", 'Items Operations'[Release type],
        "Billing type", 'Items Operations'[Billing type],
        "Operation name V2", 'Items Operations'[Operation name V2]
    )
""")

print(f"Items Operations: {len(df_items_ops):,} rows")
df_items_ops.head(3)

---
## 3. Analysis: Who Caused the Throttling?

The **#1 unanswerable question** with just Capacity Metrics. We correlate Chargeback user CU consumption with Capacity Metrics throttling windows.

In [ ]:
# ============================================================================
# ANALYSIS 1: WHO CAUSED THE THROTTLING?
#
# Logic:
# 1. From Usage Summary, find dates/hours where throttling occurred
# 2. From Chargeback, find top CU consumers on those same dates
# 3. Rank users by CU contribution during throttled periods
# ============================================================================

# Step 1: Identify throttled dates (any interactive delay or rejection > 0)
throttled_days = (
    df_usage_7d
    .assign(Date=lambda x: pd.to_datetime(x["Date"]))
    .groupby(["Capacity Id", "Date"])
    .agg({
        "Interactive rejection (s)": "sum",
        "Background rejection (s)": "sum",
        "Interactive delay (s)": "sum",
        "Maximum CU %": "max",
    })
    .reset_index()
)

throttled_days["Total throttling (s)"] = (
    throttled_days["Interactive rejection (s)"] +
    throttled_days["Background rejection (s)"] +
    throttled_days["Interactive delay (s)"]
)

throttled_days = throttled_days[throttled_days["Total throttling (s)"] > 0].copy()

if throttled_days.empty:
    print("No throttling detected in the last 7 days.")
else:
    print(f"Throttled days found: {len(throttled_days)}")

    # Step 2: For each throttled date, find top CU consumers from Chargeback
    throttled_dates_set = set(throttled_days["Date"].dt.normalize())

    cb_during_throttle = df_chargeback[
        df_chargeback["Date"].dt.normalize().isin(throttled_dates_set)
    ].copy()

    # Step 3: Rank users by CU during throttled periods
    user_cu_during_throttle = (
        cb_during_throttle
        .groupby(["Capacity Id", "User"])
        .agg(
            total_cu=pd.NamedAgg(column="CU (s)", aggfunc="sum"),
            total_ops=pd.NamedAgg(column="Operations", aggfunc="sum"),
            workspaces_used=pd.NamedAgg(column="Workspace Id", aggfunc="nunique"),
            items_used=pd.NamedAgg(column="Item Id", aggfunc="nunique"),
        )
        .reset_index()
        .sort_values("total_cu", ascending=False)
    )

    # Add throttling context
    throttle_by_cap = throttled_days.groupby("Capacity Id").agg(
        total_throttle_s=pd.NamedAgg(column="Total throttling (s)", aggfunc="sum"),
        peak_cu_pct=pd.NamedAgg(column="Maximum CU %", aggfunc="max"),
    ).reset_index()

    user_cu_during_throttle = user_cu_during_throttle.merge(throttle_by_cap, on="Capacity Id", how="left")

    # Calculate % contribution
    total_cu_per_cap = user_cu_during_throttle.groupby("Capacity Id")["total_cu"].transform("sum")
    user_cu_during_throttle["cu_pct_contribution"] = (
        user_cu_during_throttle["total_cu"] / total_cu_per_cap * 100
    ).round(2)

    print("\n=== TOP USERS DURING THROTTLED PERIODS ===")
    print(user_cu_during_throttle.head(20).to_string(index=False))

---
## 4. Analysis: Upgrade SKU or Fix Bad Actors?

The most expensive decision. We decompose CU consumption to show whether throttling is caused by broad organic growth (need more SKU) or concentrated spikes from specific items/users (fix the outliers).

In [ ]:
# ============================================================================
# ANALYSIS 2: SKU UPGRADE vs BAD ACTOR FIX
#
# Approach:
# - Compute the Gini coefficient of CU distribution across users
# - High Gini (>0.6) = concentrated load = fix the top consumers
# - Low Gini (<0.4) = distributed load = upgrade SKU
# - Also show: if top 5 users were removed, would throttling stop?
# ============================================================================

import numpy as np

def gini_coefficient(values: np.ndarray) -> float:
    """Calculate Gini coefficient (0 = perfectly equal, 1 = perfectly unequal)."""
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    if n == 0 or sorted_vals.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return float((2 * np.sum(index * sorted_vals) - (n + 1) * np.sum(sorted_vals)) / (n * np.sum(sorted_vals)))


# Per-user daily CU from Chargeback
user_daily_cu = (
    df_chargeback
    .groupby(["Capacity Id", "User"])
    .agg(total_cu=pd.NamedAgg(column="CU (s)", aggfunc="sum"))
    .reset_index()
)

for cap_id in user_daily_cu["Capacity Id"].unique():
    cap_data = user_daily_cu[user_daily_cu["Capacity Id"] == cap_id]
    cap_name = dim_capacities.loc[dim_capacities["Capacity Id"] == cap_id, "Capacity Name"].values
    cap_label = cap_name[0] if len(cap_name) > 0 else cap_id[:12]

    gini = gini_coefficient(cap_data["total_cu"].values)
    total = cap_data["total_cu"].sum()
    top5_cu = cap_data.nlargest(5, "total_cu")["total_cu"].sum()
    top5_pct = (top5_cu / total * 100) if total > 0 else 0

    print(f"\n=== Capacity: {cap_label} ===")
    print(f"  Users: {len(cap_data)} | Total CU: {total:,.0f}s")
    print(f"  Gini coefficient: {gini:.3f}")
    print(f"  Top 5 users consume: {top5_pct:.1f}% of all CU")

    if gini > 0.6:
        print(f"  >>> RECOMMENDATION: Fix the top consumers. Load is heavily concentrated.")
        print(f"      Top 5 users:")
        for _, row in cap_data.nlargest(5, "total_cu").iterrows():
            print(f"        {row['User']}: {row['total_cu']:,.0f} CU-seconds ({row['total_cu']/total*100:.1f}%)")
    elif gini > 0.4:
        print(f"  >>> RECOMMENDATION: Mixed — optimize top consumers AND consider SKU upgrade.")
    else:
        print(f"  >>> RECOMMENDATION: Upgrade SKU. Load is broadly distributed across users.")

---
## 5. Analysis: Total Cost of Ownership per Workspace

Combines compute cost (CU-seconds from Chargeback) + storage cost (GB from Capacity Metrics) into a single workspace TCO view that doesn't exist in either app.

In [ ]:
# ============================================================================
# ANALYSIS 3: TOTAL COST OF OWNERSHIP PER WORKSPACE
#
# Compute = CU-seconds from Chargeback (always available)
# Storage = GB from Capacity Metrics Storage tables (may be unavailable)
# ============================================================================

# Compute cost per workspace (Chargeback)
compute_by_ws = (
    df_chargeback
    .groupby(["Workspace Id", "Capacity Id"])
    .agg(
        compute_cu_s=pd.NamedAgg(column="CU (s)", aggfunc="sum"),
        compute_duration_s=pd.NamedAgg(column="Duration (s)", aggfunc="sum"),
        compute_operations=pd.NamedAgg(column="Operations", aggfunc="sum"),
        unique_users=pd.NamedAgg(column="User", aggfunc="nunique"),
        unique_items=pd.NamedAgg(column="Item Id", aggfunc="nunique"),
    )
    .reset_index()
)

# Storage cost per workspace (Capacity Metrics) — latest snapshot if available
if not df_storage.empty:
    latest_storage_date = df_storage["Date"].max()
    storage_by_ws = (
        df_storage[df_storage["Date"] == latest_storage_date]
        .groupby(["Workspace Id", "Capacity Id"])
        .agg(storage_gb=pd.NamedAgg(column="Utilization (GB)", aggfunc="sum"))
        .reset_index()
    )
    print(f"Storage data available (latest date: {latest_storage_date.date()})")
else:
    storage_by_ws = pd.DataFrame(columns=["Workspace Id", "Capacity Id", "storage_gb"])
    print("Storage data not available — showing compute-only TCO.")

# Merge compute + storage
tco_by_ws = compute_by_ws.merge(
    storage_by_ws, on=["Workspace Id", "Capacity Id"], how="outer"
).fillna(0)

# Add workspace names
tco_by_ws = tco_by_ws.merge(
    dim_workspaces[["Workspace Id", "Workspace Name"]].drop_duplicates(),
    on="Workspace Id", how="left"
)

tco_by_ws = tco_by_ws.sort_values("compute_cu_s", ascending=False)

display_cols = ["Workspace Name", "compute_cu_s", "unique_users", "unique_items", "compute_operations"]
if not df_storage.empty:
    display_cols.insert(2, "storage_gb")

print(f"\n=== WORKSPACE TOTAL COST OF OWNERSHIP ({len(tco_by_ws)} workspaces) ===")
print(tco_by_ws[display_cols].head(20).to_string(index=False))

---
## 6. Analysis: Stale Items Burning Capacity

Items that consume CUs (scheduled refreshes running) but have zero interactive users = pure waste.

In [ ]:
# ============================================================================
# ANALYSIS 4: STALE ITEMS — Consuming CUs but no interactive users
#
# Logic:
# 1. Flag each Chargeback row as background (refresh/pipeline/dataflow) or not
# 2. Compute bg_cu = CU where flag is True, total_cu = all CU
# 3. Items with >90% background CU and >100 CU total = stale/wasted
# ============================================================================

# Pre-compute background flag and background CU per row (clean, no fragile lambdas)
cb_flagged = df_chargeback.copy()
cb_flagged["is_background"] = cb_flagged["Operation name"].str.contains(
    "refresh|pipeline|background|dataflow", case=False, na=False
)
cb_flagged["bg_cu_val"] = cb_flagged["CU (s)"] * cb_flagged["is_background"]

item_activity = (
    cb_flagged
    .groupby(["Item Id", "Workspace Id", "Capacity Id"])
    .agg(
        total_cu=pd.NamedAgg(column="CU (s)", aggfunc="sum"),
        bg_cu=pd.NamedAgg(column="bg_cu_val", aggfunc="sum"),
        total_ops=pd.NamedAgg(column="Operations", aggfunc="sum"),
        unique_users=pd.NamedAgg(column="User", aggfunc="nunique"),
        distinct_operations=pd.NamedAgg(column="Operation name", aggfunc="nunique"),
    )
    .reset_index()
)

item_activity["interactive_cu"] = item_activity["total_cu"] - item_activity["bg_cu"]
item_activity["bg_pct"] = (item_activity["bg_cu"] / item_activity["total_cu"].replace(0, 1) * 100).round(1)

# Stale = >90% background CU and enough CU to matter
stale_items = item_activity[
    (item_activity["bg_pct"] > 90) &
    (item_activity["total_cu"] > 100)
].sort_values("total_cu", ascending=False)

# Enrich with item names
stale_items = stale_items.merge(
    dim_items[["Item Id", "Item Name", "Item Kind", "Workspace Name"]].drop_duplicates(),
    on="Item Id", how="left"
)

print(f"=== STALE ITEMS: {len(stale_items)} items consuming CU with >90% background operations ===")
print(f"Total wasted CU-seconds: {stale_items['bg_cu'].sum():,.0f}")
if not stale_items.empty:
    print(stale_items[
        ["Item Name", "Item Kind", "Workspace Name", "total_cu", "bg_cu", "interactive_cu", "bg_pct", "unique_users"]
    ].head(20).to_string(index=False))

---
## 7. Analysis: Department Impact on Capacity (Domain → Throttling)

Maps Chargeback Domains/Subdomains to Capacity Metrics throttling events. Tells you which department to call.

In [ ]:
# ============================================================================
# ANALYSIS 5: DEPARTMENT / DOMAIN IMPACT ON CAPACITY
#
# 1. Aggregate CU by Domain from Chargeback
# 2. If throttling detected (from cell 3), overlay with throttled days
# 3. Show which domains were heavy consumers during throttling
# ============================================================================

if "Domain" in df_chargeback.columns and df_chargeback["Domain"].notna().any():
    domain_cu = (
        df_chargeback
        .groupby(["Capacity Id", "Domain", "Subdomain", "Date"])
        .agg(
            daily_cu=pd.NamedAgg(column="CU (s)", aggfunc="sum"),
            daily_ops=pd.NamedAgg(column="Operations", aggfunc="sum"),
            unique_users=pd.NamedAgg(column="User", aggfunc="nunique"),
        )
        .reset_index()
    )

    # Check if throttled_days exists and has data (from cell 3)
    _has_throttling = "throttled_days" in dir() and not throttled_days.empty

    if _has_throttling:
        throttled_dates = set(throttled_days["Date"].dt.normalize())
        domain_cu["throttled_day"] = domain_cu["Date"].dt.normalize().isin(throttled_dates)

        domain_impact = (
            domain_cu
            .groupby(["Domain", "Subdomain", "throttled_day"])
            .agg(
                total_cu=pd.NamedAgg(column="daily_cu", aggfunc="sum"),
                avg_daily_cu=pd.NamedAgg(column="daily_cu", aggfunc="mean"),
                total_ops=pd.NamedAgg(column="daily_ops", aggfunc="sum"),
            )
            .reset_index()
        )

        domain_pivot = domain_impact.pivot_table(
            index=["Domain", "Subdomain"],
            columns="throttled_day",
            values=["total_cu", "avg_daily_cu"],
            fill_value=0
        )
        domain_pivot.columns = [f"{c[0]}_{'throttled' if c[1] else 'normal'}" for c in domain_pivot.columns]
        domain_pivot = domain_pivot.reset_index().sort_values("total_cu_throttled", ascending=False)

        print("=== DOMAIN IMPACT: CU DURING THROTTLED vs NORMAL DAYS ===")
        print(domain_pivot.head(15).to_string(index=False))
    else:
        domain_totals = domain_cu.groupby(["Domain", "Subdomain"]).agg(
            total_cu=pd.NamedAgg(column="daily_cu", aggfunc="sum"),
            total_ops=pd.NamedAgg(column="daily_ops", aggfunc="sum"),
            unique_users=pd.NamedAgg(column="unique_users", aggfunc="sum"),
        ).reset_index().sort_values("total_cu", ascending=False)
        print("=== DOMAIN CU CONSUMPTION (no throttling detected) ===")
        print(domain_totals.head(15).to_string(index=False))
else:
    print("No Domain data available in Chargeback (Domains may not be configured for this tenant).")

---
## 8. Analysis: Surge Protection Root Cause (v54 only)

New Surge Protection tables show blocked workspaces but not *who* triggered the surge. Cross-reference with Chargeback user data.

In [ ]:
# ============================================================================
# ANALYSIS 6: SURGE PROTECTION — WHO TRIGGERED THE BLOCK?
#
# Surge Protection tables show WHICH workspaces were blocked.
# Chargeback shows WHO was consuming CU in those workspaces leading up to it.
# ============================================================================

if not df_surge_detail.empty:
    print(f"Surge protection events: {len(df_surge_detail)}")
    
    # For each blocked workspace, find top CU consumers from Chargeback
    surge_analysis = []
    for _, surge in df_surge_detail.iterrows():
        ws_id = surge["Workspace Id"]
        blocked_date = pd.to_datetime(surge["Blocked date"]).normalize()

        # Find users in this workspace on the blocked date (and day before)
        date_range = [blocked_date - timedelta(days=1), blocked_date]
        ws_users = df_chargeback[
            (df_chargeback["Workspace Id"] == ws_id) &
            (df_chargeback["Date"].dt.normalize().isin(date_range))
        ]

        if not ws_users.empty:
            top_users = (
                ws_users.groupby("User")
                .agg(cu_total=pd.NamedAgg(column="CU (s)", aggfunc="sum"))
                .nlargest(5, "cu_total")
                .reset_index()
            )
            for _, u in top_users.iterrows():
                surge_analysis.append({
                    "Workspace": surge["Workspace name"],
                    "Blocked date": surge["Blocked date"],
                    "Duration (hrs)": surge["Blocked duration (hours)"],
                    "How Blocked": surge["How Blocked Category"],
                    "Top User": u["User"],
                    "User CU (s)": u["cu_total"],
                })

    if surge_analysis:
        df_surge_users = pd.DataFrame(surge_analysis)
        print("\n=== SURGE PROTECTION: TOP USERS IN BLOCKED WORKSPACES ===")
        print(df_surge_users.to_string(index=False))
    else:
        print("No Chargeback data found for blocked workspaces (date range mismatch).")
else:
    print("No surge protection events detected. Capacity is healthy.")

---
## 9. Analysis: Optimal Refresh Scheduling

Find low-utilization windows in Capacity Metrics and map heavy background jobs from Chargeback to suggest redistribution.

In [ ]:
# ============================================================================
# ANALYSIS 7: OPTIMAL REFRESH SCHEDULING
#
# Usage Summary (hourly grain) gives us CU utilization % by hour of day.
# We build a 24-hour heatmap showing average load and headroom per hour.
# Hours with low utilization = best windows for scheduling refreshes.
#
# Note: Chargeback data is daily grain — it cannot tell us which hour
# background jobs ran. Only Usage Summary has hourly resolution.
# ============================================================================

if not df_usage_7d.empty:
    # Hourly utilization profile
    hourly_profile = (
        df_usage_7d
        .assign(hour=lambda x: x["Timestamp"].dt.hour)
        .groupby(["Capacity Id", "hour"])
        .agg(
            avg_cu_pct=pd.NamedAgg(column="Average CU %", aggfunc="mean"),
            max_cu_pct=pd.NamedAgg(column="Maximum CU %", aggfunc="max"),
            avg_interactive_delay=pd.NamedAgg(column="Interactive delay (s)", aggfunc="mean"),
        )
        .reset_index()
    )

    # Classify hours by available headroom
    hourly_profile["headroom_pct"] = (100 - hourly_profile["avg_cu_pct"]).round(1)
    hourly_profile["window_quality"] = hourly_profile["headroom_pct"].apply(
        lambda h: "EXCELLENT" if h > 70 else "GOOD" if h > 50 else "FAIR" if h > 30 else "AVOID"
    )

    for cap_id in hourly_profile["Capacity Id"].unique():
        cap_profile = hourly_profile[hourly_profile["Capacity Id"] == cap_id]
        cap_name = dim_capacities.loc[dim_capacities["Capacity Id"] == cap_id, "Capacity Name"].values
        cap_label = cap_name[0] if len(cap_name) > 0 else cap_id[:12]

        print(f"\n=== REFRESH SCHEDULING: {cap_label} ===")
        print(cap_profile[
            ["hour", "avg_cu_pct", "max_cu_pct", "headroom_pct", "window_quality"]
        ].to_string(index=False))

        best_hours = cap_profile.nsmallest(5, "avg_cu_pct")["hour"].tolist()
        worst_hours = cap_profile.nlargest(5, "avg_cu_pct")["hour"].tolist()
        print(f"\n  BEST hours for scheduling refreshes:  {best_hours}")
        print(f"  WORST hours (avoid heavy refreshes):  {worst_hours}")
else:
    print("No usage summary data available.")

---
## 10. Analysis: Workload Mix vs Capacity Health

How does the experience/workload breakdown (Power BI, Data Engineering, Warehouse, etc.) affect capacity health?

In [ ]:
# ============================================================================
# ANALYSIS 8: WORKLOAD MIX vs CAPACITY HEALTH
#
# Chargeback has Experience codes (PBI, DW, ES, DE, DS, DF, KQL, OL)
# Usage Summary has daily CU%, throttling metrics
# Correlate: which workload types are dominant on throttled days?
# ============================================================================

EXPERIENCE_MAP = {
    "PBI": "Power BI",
    "DW": "Data Warehouse",
    "ES": "EventStream",
    "DE": "Data Engineering",
    "DS": "Data Science",
    "DF": "Data Factory",
    "KQL": "KQL Database",
    "OL": "OneLake",
}

workload_daily = (
    df_chargeback
    .assign(Experience_Name=lambda x: x["Experience"].map(EXPERIENCE_MAP).fillna(x["Experience"]))
    .groupby(["Capacity Id", "Date", "Experience_Name"])
    .agg(
        cu_total=pd.NamedAgg(column="CU (s)", aggfunc="sum"),
        ops_total=pd.NamedAgg(column="Operations", aggfunc="sum"),
        users=pd.NamedAgg(column="User", aggfunc="nunique"),
    )
    .reset_index()
)

# Overall workload mix
workload_totals = (
    workload_daily
    .groupby("Experience_Name")
    .agg(
        total_cu=pd.NamedAgg(column="cu_total", aggfunc="sum"),
        total_ops=pd.NamedAgg(column="ops_total", aggfunc="sum"),
    )
    .reset_index()
    .sort_values("total_cu", ascending=False)
)
workload_totals["cu_pct"] = (workload_totals["total_cu"] / workload_totals["total_cu"].sum() * 100).round(1)

print("=== WORKLOAD MIX (all days) ===")
print(workload_totals.to_string(index=False))

# Compare throttled vs normal days
if not throttled_days.empty:
    throttled_dates = set(throttled_days["Date"].dt.normalize())
    workload_daily["throttled"] = workload_daily["Date"].dt.normalize().isin(throttled_dates)

    mix_comparison = (
        workload_daily
        .groupby(["Experience_Name", "throttled"])
        .agg(avg_daily_cu=pd.NamedAgg(column="cu_total", aggfunc="mean"))
        .reset_index()
        .pivot_table(index="Experience_Name", columns="throttled", values="avg_daily_cu", fill_value=0)
    )
    mix_comparison.columns = ["Normal days", "Throttled days"]
    mix_comparison["Spike ratio"] = (mix_comparison["Throttled days"] / mix_comparison["Normal days"].replace(0, 1)).round(2)
    mix_comparison = mix_comparison.sort_values("Spike ratio", ascending=False)

    print("\n=== WORKLOAD CU: THROTTLED vs NORMAL DAYS ===")
    print("(Spike ratio > 1.5 = this workload increases significantly on throttled days)")
    print(mix_comparison.to_string())

---
## 11. Analysis: Item History Trending (v54 — Throttling per Item over Time)

New Item History tables let us trend CU consumption and throttling per item. Identify items with *growing* throttling problems.

In [ ]:
# ============================================================================
# ANALYSIS 9: ITEM THROTTLING TRENDS (Item History — v54)
#
# Identify items where throttling is INCREASING over time.
# Item History tables may be empty if the app's date range parameters
# (StartHistory/EndHistory) haven't been set.
# ============================================================================

if not df_item_history.empty and "Throttling (s)" in df_item_history.columns:
    # Aggregate per item per day
    item_daily = (
        df_item_history
        .groupby(["ArtifactName", "WorkspaceName", "ArtifactKind", "Experience", "Day"])
        .agg(
            cu_s=pd.NamedAgg(column="CU (s)", aggfunc="sum"),
            throttle_s=pd.NamedAgg(column="Throttling (s)", aggfunc="sum"),
            ops=pd.NamedAgg(column="Operations", aggfunc="sum"),
        )
        .reset_index()
    )

    # Find items with any throttling
    items_with_throttle = (
        item_daily[item_daily["throttle_s"] > 0]
        .groupby(["ArtifactName", "WorkspaceName", "ArtifactKind"])
        .agg(
            total_throttle_s=pd.NamedAgg(column="throttle_s", aggfunc="sum"),
            total_cu_s=pd.NamedAgg(column="cu_s", aggfunc="sum"),
            days_throttled=pd.NamedAgg(column="Day", aggfunc="nunique"),
            total_ops=pd.NamedAgg(column="ops", aggfunc="sum"),
        )
        .reset_index()
        .sort_values("total_throttle_s", ascending=False)
    )

    print(f"=== ITEMS WITH THROTTLING: {len(items_with_throttle)} ===")
    if not items_with_throttle.empty:
        print(items_with_throttle.head(20).to_string(index=False))

        # Trend: compare first half vs second half of date range
        unique_days = sorted(item_daily["Day"].unique())
        if len(unique_days) >= 4:
            mid_date = unique_days[len(unique_days) // 2]
            item_daily["period"] = item_daily["Day"].apply(lambda d: "recent" if d >= mid_date else "earlier")

            trend = (
                item_daily[item_daily["throttle_s"] > 0]
                .groupby(["ArtifactName", "period"])
                .agg(avg_throttle=pd.NamedAgg(column="throttle_s", aggfunc="mean"))
                .reset_index()
                .pivot_table(index="ArtifactName", columns="period", values="avg_throttle", fill_value=0)
            )
            if "earlier" in trend.columns and "recent" in trend.columns:
                trend["change_pct"] = ((trend["recent"] - trend["earlier"]) / trend["earlier"].replace(0, 1) * 100).round(1)
                worsening = trend[trend["change_pct"] > 20].sort_values("change_pct", ascending=False)
                if not worsening.empty:
                    print("\n=== ITEMS WITH WORSENING THROTTLING (>20% increase) ===")
                    print(worsening.head(10).to_string())
                else:
                    print("\nNo items with worsening throttling trend detected.")
    else:
        print("No items experienced throttling in this period.")
else:
    print("Item History data not available (tables may require date range parameters to be set in the app).")

---
## 12. Executive Summary Dashboard Data

Pre-computed metrics ready for Power BI, Excel, or any downstream visualization.

In [ ]:
# ============================================================================
# EXECUTIVE SUMMARY — Key metrics for each capacity
# ============================================================================

for cap_id in dim_capacities["Capacity Id"].unique():
    cap_row = dim_capacities[dim_capacities["Capacity Id"] == cap_id].iloc[0]
    cap_name = cap_row.get("Capacity Name", cap_id[:12])
    sku = cap_row.get("SKU", "N/A")
    region = cap_row.get("Region", "N/A")
    cores = cap_row.get("Core count", "N/A")

    # Chargeback totals
    cb_cap = df_chargeback[df_chargeback["Capacity Id"] == cap_id]
    total_cu = cb_cap["CU (s)"].sum()
    total_ops = cb_cap["Operations"].sum()
    total_users = cb_cap["User"].nunique()
    total_items = cb_cap["Item Id"].nunique()
    total_workspaces = cb_cap["Workspace Id"].nunique()

    # Usage summary (7d)
    usage_cap = df_usage_7d[df_usage_7d["Capacity Id"] == cap_id]
    avg_util = usage_cap["Average CU %"].mean() if not usage_cap.empty else 0
    peak_util = usage_cap["Maximum CU %"].max() if not usage_cap.empty else 0
    total_rejection = (
        usage_cap["Interactive rejection (s)"].sum() +
        usage_cap["Background rejection (s)"].sum()
    ) if not usage_cap.empty else 0

    # Storage (may be unavailable)
    if not df_storage.empty:
        stor_cap = df_storage[df_storage["Capacity Id"] == cap_id]
        latest_storage = stor_cap.groupby("Workspace Id")["Utilization (GB)"].last().sum() if not stor_cap.empty else 0
    else:
        latest_storage = None

    # Surge
    surge_count = len(df_surge_detail[df_surge_detail["Capacity Id"] == cap_id]) if not df_surge_detail.empty else 0

    print(f"\n{'='*70}")
    print(f"CAPACITY: {cap_name}")
    print(f"{'='*70}")
    print(f"  SKU: {sku} | Region: {region} | Cores: {cores}")
    print(f"")
    print(f"  COMPUTE (Chargeback period):")
    print(f"    Total CU consumed:    {total_cu:>15,.0f} seconds")
    print(f"    Total operations:     {total_ops:>15,.0f}")
    print(f"    Unique users:         {total_users:>15,}")
    print(f"    Unique items:         {total_items:>15,}")
    print(f"    Workspaces:           {total_workspaces:>15,}")
    print(f"")
    print(f"  HEALTH (Last 7 days):")
    print(f"    Avg utilization:      {avg_util:>14.1f}%")
    print(f"    Peak utilization:     {peak_util:>14.1f}%")
    print(f"    Total rejection:      {total_rejection:>15,.0f} seconds")
    print(f"    Surge blocks:         {surge_count:>15,}")
    if latest_storage is not None:
        print(f"")
        print(f"  STORAGE:")
        print(f"    Total utilization:    {latest_storage:>14.2f} GB")

---
## 13. Export Unified Dataset

Export the joined datasets for downstream consumption — Power BI, Lakehouse, or any BI tool.

In [ ]:
# ============================================================================
# EXPORT — Save unified datasets for downstream use
# ============================================================================

import os

export_dir = "monitoring_exports"
os.makedirs(export_dir, exist_ok=True)

exports = {
    "dim_capacities": dim_capacities,
    "dim_items": dim_items,
    "dim_workspaces": dim_workspaces,
    "dim_domains": dim_domains,
    "fact_chargeback": df_chargeback,
    "fact_usage_summary_7d": df_usage_7d,
    "fact_usage_operations_7d": df_usage_ops_7d,
    "fact_storage_daily": df_storage,
    "fact_surge_protection": df_surge_detail,
    "fact_item_history": df_item_history,
    "fact_items_operations": df_items_ops,
    "analysis_workspace_tco": tco_by_ws,
}

for name, df in exports.items():
    if df is not None and not df.empty:
        path = os.path.join(export_dir, f"{name}.csv")
        df.to_csv(path, index=False)
        print(f"  Exported {name}: {len(df):,} rows -> {path}")
    else:
        print(f"  Skipped {name} (empty or unavailable)")

print(f"\nAll exports saved to {os.path.abspath(export_dir)}/")
print(f"\nThese CSVs can be imported into a Fabric Lakehouse, loaded into")
print(f"Power BI, or used as input for scheduled monitoring pipelines.")

---
## Schema Reference

### Capacity Metrics App (v54) — 105 Tables, 294 Measures

**New table families in v54 (not in v28):**

| Family | Tables | Purpose |
|--------|--------|---------|
| **Item History** | `Item History Main`, `Operation`, `Operation Detail`, `Summary` + 7 filter tables | Per-item CU/duration/throttling over configurable date range |
| **Usage Summary/Operation** | 1hr, 24hr, 7d, 14d, 30d variants | Pre-aggregated capacity health snapshots at multiple time windows |
| **Surge Protection** | `By Day`, `By Hour`, `Blocked Workspaces Detail` | Workspace-level surge blocking with duration, affected users, rejection counts |
| **Workload Autoscale** | 10 tables for CU/metrics/timepoint per workload | Per-workload autoscale limit tracking |
| **Timepoint Item Detail** | `Interactive Item Detail`, `Background Item Detail` | Per-operation-ID detail within a timepoint |
| **Timepoint Summaries** | `Interactive Summary`, `Background Summary` | Aggregated CU/duration/throttling/operations per timepoint |
| **Items Operations** | 1 table, 17 columns | Operation-level metadata per item including Release type |

### Chargeback App — 10 Tables, 31 Measures

**New in latest version:**

| Table | Change |
|-------|--------|
| `Domains` | Added `Domain Id`, `Subdomain Id` (was just name fields) |
| `Days Ago to Start` | New parameter table for configurable date range |
| `Export page optional columns` | New UX support table |